# CB-LLM Generation — Evaluation Notebook

**Paper:** [Concept Bottleneck Large Language Models](https://arxiv.org/abs/2412.07992) (ICLR 2025)

This notebook evaluates pre-trained CB-LLM generation models (LoRA adapters + CBL) from HuggingFace. It covers:

1. **Concept Detection** — measures whether the CBL correctly predicts the class concept at each token position
2. **Controlled Generation** — generates text by intervening on specific concept neurons
3. **Steerability** — quantifies how reliably concept interventions steer the generated text
4. **Perplexity** — measures fluency of generated text using Llama-3-8B as the reference model
5. **Concept Weight Visualization** — shows which vocabulary tokens are most associated with each concept
6. **Per-word Activation Detection** — tracks concept activation across token positions in a given input

**Runtime requirement:** A100 GPU (≥40 GB VRAM). Set via *Runtime → Change runtime type → A100*.

## 0. Environment Setup

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies
# (Colab ships with torch; we only need the extra packages)
!pip install -q \
    transformers==4.44.0 \
    datasets==3.0.2 \
    peft==0.13.2 \
    evaluate==0.4.3 \
    huggingface_hub \
    langdetect \
    accelerate

print('Installation complete.')

In [ ]:
# Clone the CB-LLMs repository so we can import config / modules / utils
import os

REPO_URL = 'https://github.com/lilywenglab/CB-LLMs.git'  # adjust if using a fork

if not os.path.isdir('/content/CB-LLMs'):
    !git clone --quiet {REPO_URL} /content/CB-LLMs
    print('Repo cloned.')
else:
    print('Repo already present.')

os.chdir('/content/CB-LLMs/generation')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Authenticate with HuggingFace (needed for Meta-Llama-3-8B and private team repos)
from huggingface_hub import login

# Paste your HF read-token here, OR store it as a Colab secret named HF_TOKEN
HF_TOKEN = ''   # e.g. 'hf_XXXX'  — leave empty to use the Colab Secrets approach

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        print('Token loaded from Colab secrets.')
    except Exception:
        print('No token found. Use login() interactively or set HF_TOKEN above.')

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged in to HuggingFace.')

## 1. Configuration

Edit the cells below to select the dataset and point to your team's HuggingFace model repository.

In [ ]:
# ── USER SETTINGS ──────────────────────────────────────────────────────────────

# Dataset to evaluate. Choices:
#   'SetFit/sst2'    → 2 concepts : negative, positive
#   'yelp_polarity'  → 2 concepts : negative, positive
#   'ag_news'        → 4 concepts : world, sports, business, technology
#   'dbpedia_14'     → 14 concepts (company, education, artist, …)
DATASET = 'SetFit/sst2'

# HuggingFace repo that contains the pre-trained LoRA adapters + CBL weights.
# Expected layout inside the repo:
#   from_pretained_llama3_lora_cbm/<dataset_name>/llama3/   ← PEFT adapter
#   from_pretained_llama3_lora_cbm/<dataset_name>/cbl.pt    ← CBL weights
HF_MODEL_REPO = 'cesun/cbllm-generation'   # replace with your team's repo

# Where to download / cache the model files on the Colab VM
MODEL_DIR = '/content/models'

# Generation hyper-parameters
GEN_LENGTH         = 100    # tokens to generate
GEN_TEMP           = 0.7
GEN_TOP_K          = 100
GEN_TOP_P          = 0.9
GEN_REP_PENALTY    = 1.5
INTERVENTION_VALUE = 150 if DATASET == 'dbpedia_14' else 100

# Concept-detection evaluation
EVAL_BATCH_SIZE = 8
EVAL_MAX_LENGTH = 350

# Steerability: number of generation rounds per concept
STEER_ROUNDS = 10   # total samples = STEER_ROUNDS * n_concepts

# Perplexity: number of samples
PPLX_SAMPLES = 100

# ───────────────────────────────────────────────────────────────────────────────

import sys, os
sys.path.insert(0, '/content/CB-LLMs/generation')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import config as CFG
CONCEPT_SET = CFG.concepts_from_labels[DATASET]
N_CONCEPTS  = len(CONCEPT_SET)
DATASET_KEY = DATASET.replace('/', '_')

print(f'Dataset   : {DATASET}')
print(f'Concepts  : {CONCEPT_SET}')
print(f'HF repo   : {HF_MODEL_REPO}')

## 2. Download Pre-trained Models

In [ ]:
from huggingface_hub import snapshot_download
import shutil, pathlib

# Download only the files relevant to the chosen dataset to save time
allow_pattern = f'from_pretained_llama3_lora_cbm/{DATASET_KEY}/**'

print(f'Downloading model files from {HF_MODEL_REPO} …')
snapshot_download(
    repo_id=HF_MODEL_REPO,
    local_dir=MODEL_DIR,
    allow_patterns=[allow_pattern],
    token=HF_TOKEN or None,
)
print('Download complete.')

# Paths that the existing scripts expect (relative to generation/)
PEFT_PATH = os.path.join(MODEL_DIR,
    f'from_pretained_llama3_lora_cbm/{DATASET_KEY}/llama3')
CBL_PATH  = os.path.join(MODEL_DIR,
    f'from_pretained_llama3_lora_cbm/{DATASET_KEY}/cbl.pt')

assert os.path.isdir(PEFT_PATH), f'PEFT adapter not found at {PEFT_PATH}'
assert os.path.isfile(CBL_PATH), f'CBL weights not found at {CBL_PATH}'
print(f'PEFT adapter : {PEFT_PATH}')
print(f'CBL weights  : {CBL_PATH}')

## 3. Load Backbone + LoRA Adapter + CBL

In [ ]:
import torch
from transformers import LlamaConfig, LlamaModel, AutoTokenizer
from modules import CBL

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

BASE_LM = 'meta-llama/Meta-Llama-3-8B'

print('Loading Llama config and tokenizer …')
llama_config = LlamaConfig.from_pretrained(BASE_LM)
tokenizer    = AutoTokenizer.from_pretrained(BASE_LM)
tokenizer.pad_token = tokenizer.eos_token

print('Loading Llama-3-8B backbone (bfloat16) …')
preLM = LlamaModel.from_pretrained(BASE_LM, torch_dtype=torch.bfloat16).to(device)
preLM.load_adapter(PEFT_PATH)
preLM.eval()
print('Backbone + LoRA ready.')

print('Loading CBL …')
cbl = CBL(llama_config, N_CONCEPTS, tokenizer).to(device)
cbl.load_state_dict(torch.load(CBL_PATH, map_location=device))
cbl.eval()
print('CBL ready.')

EOS_TOKEN_ID = tokenizer.eos_token_id  # 128001 for Llama-3

---
## Evaluation 1 — Concept Detection

Measures how accurately the CBL predicts the ground-truth class label (as a concept) at each token position on the held-out test set. Mirrors `test_concepts.py`.

In [ ]:
import numpy as np
from datasets import load_dataset
import evaluate
from utils import eos_pooling

# ── Dataset ──────────────────────────────────────────────────────────────────
print('Loading test dataset …')
test_dataset = load_dataset(DATASET, split='test')
print(f'Test set size: {len(test_dataset)}')

if DATASET == 'ag_news':
    def clean_agnews(example):
        example['text'] = example['text'].replace('#36;', '').replace('#39;', "'")
        return example
    test_dataset = test_dataset.map(clean_agnews)

example_field = CFG.example_name[DATASET]
encoded_test = test_dataset.map(
    lambda e: tokenizer(
        e[example_field], padding=True, truncation=True, max_length=EVAL_MAX_LENGTH
    ),
    batched=True, batch_size=len(test_dataset)
)
cols_to_drop = [example_field]
if DATASET == 'SetFit/sst2':  cols_to_drop.append('label_text')
if DATASET == 'dbpedia_14':   cols_to_drop.append('title')
encoded_test = encoded_test.remove_columns(cols_to_drop)
encoded_test_dict = encoded_test[:len(encoded_test)]

# ── DataLoader ────────────────────────────────────────────────────────────────
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, enc):
        self.enc = enc
    def __len__(self):
        return len(self.enc['input_ids'])
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.enc.items()}

test_loader = torch.utils.data.DataLoader(
    TextDataset(encoded_test_dict),
    batch_size=EVAL_BATCH_SIZE, shuffle=False
)

# ── Inference ─────────────────────────────────────────────────────────────────
print('Running concept detection …')
metric = evaluate.load('accuracy')
all_preds = []

for batch in test_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        features = preLM(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask']
        ).last_hidden_state
        concepts, _, _ = cbl(features.float())
    all_preds.append(eos_pooling(concepts, batch['attention_mask']))

all_preds = torch.cat(all_preds, dim=0).detach().cpu()
pred_labels = np.argmax(all_preds.numpy(), axis=-1)
true_labels = encoded_test_dict['label']
metric.add_batch(predictions=pred_labels, references=true_labels)
result = metric.compute()

print()
print('=' * 40)
print(f'Concept Detection Accuracy: {result["accuracy"]:.4f}')
print('=' * 40)

---
## Evaluation 2 — Controlled Generation

Demonstrates steerable generation by setting one concept neuron to a large positive value and leaving all others at zero. Mirrors `test_generation.py`.

In [ ]:
print('Concept set:', CONCEPT_SET)
print(f'Intervention value: {INTERVENTION_VALUE}')
print()

# Empty prompt — generation starts from scratch
input_ids = torch.tensor([tokenizer.encode('')]).to(device)

for concept_idx, concept_name in enumerate(CONCEPT_SET):
    v = [0] * N_CONCEPTS
    v[concept_idx] = INTERVENTION_VALUE

    with torch.no_grad():
        text_ids, concept_activations = cbl.generate(
            input_ids, preLM,
            intervene=v,
            length=GEN_LENGTH,
            temp=GEN_TEMP,
            topk=GEN_TOP_K,
            topp=GEN_TOP_P,
            repetition_penalty=GEN_REP_PENALTY,
            eos_token_id=EOS_TOKEN_ID,
        )

    # Strip BOS/EOS tokens before decoding
    special_ids = torch.tensor([tokenizer.bos_token_id, tokenizer.eos_token_id]).to(device)
    text_ids_clean = text_ids[0][~torch.isin(text_ids[0], special_ids)]
    generated_text = tokenizer.decode(text_ids_clean)

    print(f'── Concept: [{concept_name.upper()}] (neuron {concept_idx}, v={INTERVENTION_VALUE}) ──')
    print(generated_text)
    print()

---
## Evaluation 3 — Steerability

Quantifies how reliably concept interventions steer the generated text. A RoBERTa classifier (fine-tuned on the dataset) judges whether each generated sample matches the intended concept.

Mirrors `train_classifier.py` + `test_steerability.py`.

In [ ]:
# ── Train (or load) the steerability classifier ───────────────────────────────
from transformers import RobertaTokenizerFast
from modules import Roberta_classifier

CLASSIFIER_PATH = os.path.join(MODEL_DIR, f'{DATASET_KEY}_classifier.pt')

roberta_tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base')
classifier = Roberta_classifier(N_CONCEPTS).to(device)

# Try to download the classifier from the same HF repo first
classifier_trained = False
try:
    from huggingface_hub import hf_hub_download
    clf_file = hf_hub_download(
        repo_id=HF_MODEL_REPO,
        filename=f'{DATASET_KEY}_classifier.pt',
        local_dir=MODEL_DIR,
        token=HF_TOKEN or None,
    )
    classifier.load_state_dict(torch.load(clf_file, map_location=device))
    classifier_trained = True
    print(f'Classifier loaded from HF repo: {clf_file}')
except Exception as e:
    print(f'Classifier not found in HF repo ({e}). Will train from scratch.')

if not classifier_trained:
    # ── Train RoBERTa classifier on the training split ─────────────────────
    print('Training steerability classifier (RoBERTa fine-tune) …')
    train_split = load_dataset(DATASET, split='train')

    if DATASET == 'ag_news':
        train_split = train_split.map(clean_agnews)

    enc_train = train_split.map(
        lambda e: roberta_tokenizer(
            e[example_field], padding=True, truncation=True, max_length=100
        ),
        batched=True, batch_size=len(train_split)
    )
    cols_to_drop_clf = [example_field]
    if DATASET == 'SetFit/sst2':  cols_to_drop_clf.append('label_text')
    if DATASET == 'dbpedia_14':   cols_to_drop_clf.append('title')
    enc_train = enc_train.remove_columns(cols_to_drop_clf)
    enc_train_dict = enc_train[:len(enc_train)]

    clf_loader = torch.utils.data.DataLoader(
        TextDataset(enc_train_dict), batch_size=16, shuffle=True
    )
    optimizer_clf = torch.optim.Adam(classifier.parameters(), lr=5e-6)
    clf_epochs = CFG.epoch[DATASET]

    for ep in range(clf_epochs):
        classifier.train()
        losses = []
        for i, batch in enumerate(clf_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = classifier(batch)
            loss = torch.nn.CrossEntropyLoss()(logits, batch['label'])
            optimizer_clf.zero_grad()
            loss.backward()
            optimizer_clf.step()
            losses.append(loss.item())
            if i % 100 == 0:
                print(f'  Epoch {ep+1} | batch {i} | loss {loss.item():.4f}', end='\r')
        print(f'  Epoch {ep+1} | avg loss {sum(losses)/len(losses):.4f}' + ' ' * 30)

    torch.save(classifier.state_dict(), CLASSIFIER_PATH)
    print(f'Classifier saved to {CLASSIFIER_PATH}')

classifier.eval()
print('Classifier ready.')

In [ ]:
# ── Steerability evaluation ───────────────────────────────────────────────────
import gc

print(f'Generating {STEER_ROUNDS} rounds × {N_CONCEPTS} concepts = {STEER_ROUNDS * N_CONCEPTS} samples …')

acc_metric = evaluate.load('accuracy')
all_steer_preds = []
all_steer_refs  = []

input_ids_empty = torch.tensor([tokenizer.encode('')]).to(device)
special_ids = torch.tensor([tokenizer.bos_token_id, tokenizer.eos_token_id]).to(device)

for round_idx in range(STEER_ROUNDS):
    print(f'  Round {round_idx + 1}/{STEER_ROUNDS}', end='\r')
    for j, concept_name in enumerate(CONCEPT_SET):
        v = [0] * N_CONCEPTS
        v[j] = INTERVENTION_VALUE

        with torch.no_grad():
            text_ids, _ = cbl.generate(
                input_ids_empty, preLM,
                intervene=v,
                length=GEN_LENGTH,
                temp=GEN_TEMP,
                topk=GEN_TOP_K,
                topp=GEN_TOP_P,
                repetition_penalty=GEN_REP_PENALTY,
                eos_token_id=EOS_TOKEN_ID,
            )

        text_clean = tokenizer.decode(
            text_ids[0][~torch.isin(text_ids[0], special_ids)]
        )

        rob_ids = torch.tensor(
            [roberta_tokenizer.encode(text_clean, max_length=512, truncation=True)]
        ).to(device)
        rob_mask = torch.ones_like(rob_ids)

        with torch.no_grad():
            logits = classifier({'input_ids': rob_ids, 'attention_mask': rob_mask})

        all_steer_preds.append(logits.detach().cpu())
        all_steer_refs.append(j)

all_steer_preds = torch.cat(all_steer_preds, dim=0)
pred_labels_steer = np.argmax(all_steer_preds.numpy(), axis=-1)

acc_metric.add_batch(predictions=pred_labels_steer, references=all_steer_refs)
steer_result = acc_metric.compute()

print()
print('=' * 40)
print(f'Steerability Accuracy: {steer_result["accuracy"]:.4f}')
print('=' * 40)

# Per-concept breakdown
pred_arr = np.array(pred_labels_steer)
ref_arr  = np.array(all_steer_refs)
print('\nPer-concept steerability:')
for j, name in enumerate(CONCEPT_SET):
    mask = ref_arr == j
    acc_j = (pred_arr[mask] == j).mean()
    print(f'  [{name:>15}]: {acc_j:.4f}  ({mask.sum()} samples)')

---
## Evaluation 4 — Perplexity

Generates text **without** any concept intervention and measures fluency via Llama-3-8B perplexity. Lower is better. Mirrors `test_perplexity.py`.

> Note: The perplexity metric reloads Llama-3-8B internally, so we first free the backbone from GPU memory.

In [ ]:
# Generate PPLX_SAMPLES texts (no intervention)
print(f'Generating {PPLX_SAMPLES} samples for perplexity evaluation …')
input_ids_empty = torch.tensor([tokenizer.encode('')]).to(device)
generated_texts = []

for i in range(PPLX_SAMPLES):
    print(f'  Sample {i+1}/{PPLX_SAMPLES}', end='\r')
    with torch.no_grad():
        text_ids, _ = cbl.generate(
            input_ids_empty, preLM,
            intervene=None,
            length=GEN_LENGTH,
            temp=GEN_TEMP,
            topk=GEN_TOP_K,
            topp=GEN_TOP_P,
            repetition_penalty=GEN_REP_PENALTY,
            eos_token_id=EOS_TOKEN_ID,
        )
    generated_texts.append(tokenizer.decode(text_ids[0], skip_special_tokens=True))

print(f'\nGenerated {len(generated_texts)} texts.')
print('Sample preview (first 3):')
for t in generated_texts[:3]:
    print(' •', t[:120], '…')

In [ ]:
# Free the CB-LLM from GPU before loading the perplexity reference model
del preLM, cbl, classifier
gc.collect()
torch.cuda.empty_cache()
print('GPU memory freed.')

# Compute perplexity using Llama-3-8B as the reference model
print('Computing perplexity (this reloads Llama-3-8B internally) …')
perplexity_metric = evaluate.load('perplexity', module_type='metric')
perplexity_metric.add_batch(predictions=generated_texts)
pplx_result = perplexity_metric.compute(
    model_id=BASE_LM,
    max_length=GEN_LENGTH,
    # batch_size can be set lower if OOM; A100-40GB can handle 8
    batch_size=8,
)

print()
print('=' * 40)
print(f'Mean Perplexity : {pplx_result["mean_perplexity"]:.2f}')
print('=' * 40)

---
## Evaluation 5 — Concept Weight Visualization

For each concept neuron, shows the 10 vocabulary tokens with the highest connection weight in the CBL's final linear layer. Mirrors `test_weight.py`.

Paste the printed output into [SankeyMATIC](https://sankeymatic.com/build/) to create a Sankey diagram.

In [ ]:
# Reload the models (freed above for perplexity)
print('Reloading backbone and CBL …')
gc.collect()
torch.cuda.empty_cache()

preLM = LlamaModel.from_pretrained(BASE_LM, torch_dtype=torch.bfloat16).to(device)
preLM.load_adapter(PEFT_PATH)
preLM.eval()

cbl = CBL(llama_config, N_CONCEPTS, tokenizer).to(device)
cbl.load_state_dict(torch.load(CBL_PATH, map_location=device))
cbl.eval()
print('Models ready.')

In [ ]:
# Concept → top-10 token weights
# Shape of cbl.fc.weight: (vocab_size, concept_dim + 768)
# The concept slice is the first `N_CONCEPTS` columns when transposed
w = cbl.fc.weight.data[:, :N_CONCEPTS].T   # (N_CONCEPTS, vocab_size)

TOP_K_TOKENS = 10
print(f'Top-{TOP_K_TOKENS} tokens per concept (format compatible with SankeyMATIC):\n')

for i, concept_name in enumerate(CONCEPT_SET):
    top_vals, top_ids = torch.topk(w[i], k=TOP_K_TOKENS)
    print(f'Concept: {concept_name}')
    for j in range(TOP_K_TOKENS):
        token_str = tokenizer.decode([top_ids[j].item()]).strip()
        weight_val = round(float(top_vals[j].detach().cpu()), 4)
        print(f'  {concept_name} [{weight_val}] {token_str}')
    print()

# Sparsity: fraction of concept weights above threshold
sparsity = float((w > 1e-6).count_nonzero()) / w.numel()
print(f'Concept weight sparsity (>1e-6): {sparsity:.4f}')

---
## Evaluation 6 — Per-word Concept Activation Detection

Tracks how each concept neuron activates as the model reads successive tokens of a given input sentence. Mirrors `test_detection.py`.

In [ ]:
import torch.nn.functional as F

# ── Input text for activation tracing ─────────────────────────────────────────
# Change this to any text you want to analyse.
PROBE_TEXT = (
    'I went there today! The cut was terrible! '
    'I have an awful experience. They lady that cut my hair was nice '
    'but she wanted to leave early so she made a disaster in my head!'
)

# ── Forward pass at each prefix length ────────────────────────────────────────
probe_ids = torch.tensor([tokenizer.encode(PROBE_TEXT)]).to(device)
seq_len   = probe_ids.shape[1]
activations = []

with torch.no_grad():
    for i in range(seq_len):
        feats    = preLM(input_ids=probe_ids[:, :i + 1]).last_hidden_state
        concepts, _, _ = cbl(feats.float())
        # Softmax over concept dimension at the last token position
        activations.append(F.softmax(concepts[0, -1, :], dim=-1).detach().cpu())

activations = torch.stack(activations, dim=0)   # (seq_len, N_CONCEPTS)

# ── Map sub-word tokens → words ───────────────────────────────────────────────
encoded_probe = tokenizer(PROBE_TEXT)
token_word_map = []
for word_id in encoded_probe.word_ids():
    if word_id is not None:
        start, end = encoded_probe.word_to_tokens(word_id)
        span = [start, end - 1]
        if not token_word_map or token_word_map[-1] != span:
            token_word_map.append(span)

all_tokens = [tokenizer.decode(t) for t in probe_ids[0]]
words = [''.join(all_tokens[m[0]: m[1] + 1]) for m in token_word_map]

# ── Print per-word concept probabilities ──────────────────────────────────────
print('Words:', words)
print()
for i, concept_name in enumerate(CONCEPT_SET):
    a = activations[:, i]
    probs = [round(float(a[m[0]: m[1] + 1].mean()), 3) for m in token_word_map]
    print(f'{concept_name:>12}: {probs}')

In [ ]:
# Optional: heat-map plot of per-word activations
try:
    import matplotlib.pyplot as plt
    import matplotlib

    heat_data = []
    for i in range(N_CONCEPTS):
        a = activations[:, i]
        row = [float(a[m[0]: m[1] + 1].mean()) for m in token_word_map]
        heat_data.append(row)

    fig, ax = plt.subplots(figsize=(max(12, len(words)), max(2, N_CONCEPTS)))
    im = ax.imshow(heat_data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(words)))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(N_CONCEPTS))
    ax.set_yticklabels(CONCEPT_SET)
    ax.set_title(f'Concept Activation Heatmap — {DATASET}')
    plt.colorbar(im, ax=ax, label='Softmax probability')
    plt.tight_layout()
    plt.savefig('/content/activation_heatmap.png', dpi=150)
    plt.show()
    print('Heatmap saved to /content/activation_heatmap.png')
except ImportError:
    print('matplotlib not available; skipping heatmap.')

---
## Summary

The cell below collects all metrics computed in this notebook and prints a results table.

In [ ]:
# Results will only show metrics that were computed above.
# Re-run each section to refresh.

results = {}

try:
    results['Concept Detection Accuracy'] = f"{result['accuracy']:.4f}"
except NameError:
    results['Concept Detection Accuracy'] = 'not computed'

try:
    results['Steerability Accuracy'] = f"{steer_result['accuracy']:.4f}"
except NameError:
    results['Steerability Accuracy'] = 'not computed'

try:
    results['Mean Perplexity'] = f"{pplx_result['mean_perplexity']:.2f}"
except NameError:
    results['Mean Perplexity'] = 'not computed'

print(f'\n{"="*50}')
print(f'CB-LLM Generation Results  |  Dataset: {DATASET}')
print(f'{"="*50}')
for k, v in results.items():
    print(f'  {k:<35} {v}')
print(f'{"="*50}')
print()
print('Reference numbers from the CB-LLM paper (SST2):')
print('  Concept Detection Accuracy  :  0.9638')
print('  Steerability Accuracy       :  0.82')
print('  Mean Perplexity             :  116.22')